# Единая методика финансового эффекта

Артефакты в `data/`:

- общие (корень `data/`): `fin_effect_snapshots.csv`, `fin_effect_weekly.html`;
- на каждый прогон — папка `data/YYYY-MM-DD_HHMM/` с plan / report / conclusion
  и Excel-аудитом с формулами.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_here, *_here.parents) if (path / "pyproject.toml").exists()
)
for path in (PROJECT_ROOT / "src", PROJECT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from querulus.fin_effect.excel_monitoring import (
    RETRO_AS_OF_DEFAULT,
    VITRINA_TABLE_DEFAULT,
    estimate_monitoring_effect,
    load_monitoring_frame,
)
from querulus.fin_effect.monitoring_excel_audit import export_monitoring_audit_xlsx
from querulus.fin_effect.monitoring_report import (
    dated_artifact_names,
    export_run_dir,
    report_export_stamp,
    write_all_monitoring_htmls,
    write_error_html,
)
from querulus.fin_effect.monitoring_snapshots import (
    PILOT_START_DEFAULT,
    SNAPSHOT_FILENAME,
    WEEKLY_HTML_FILENAME,
    append_snapshot_log,
    run_weekly_monitoring_series,
    save_weekly_outputs,
)
from querulus.fin_effect.psr_development import compute_psr_development_lags

DATA_DIR = PROJECT_ROOT / "monitoring" / "fin_effects" / "data"
SNAPSHOT_CSV = DATA_DIR / SNAPSHOT_FILENAME
WEEKLY_HTML = DATA_DIR / WEEKLY_HTML_FILENAME
RETRO_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/processed/querulus_train_dataset.parquet"
)
PRETENSIONS_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/raw/df_pretensions.parquet"
)
CLAIMS_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/raw/df_claims_incoming.parquet"
)
LOOKBACK_YEARS = 2.0
RETRO_AS_OF = RETRO_AS_OF_DEFAULT
T_CALC = None
DISCOUNT_RATE = 0.12
RESIDUAL_SHARE = 0.07
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_COMPLIANCE_ITERATIONS = 200
BOOTSTRAP_N_JOBS = -1
WRITE_AUDIT_XLSX = True
RUN_WEEKLY_SERIES = True
WEEKLY_BOOTSTRAP_ITERATIONS = 0
PILOT_START = PILOT_START_DEFAULT


In [ ]:
export_stamp = report_export_stamp()
run_dir = export_run_dir(DATA_DIR, export_stamp)
artifact_names = dated_artifact_names(export_stamp)
plan_path = run_dir / artifact_names["plan"]
report_path = run_dir / artifact_names["report"]
conclusion_path = run_dir / artifact_names["conclusion"]
audit_path = run_dir / artifact_names["audit"]
weekly_html = WEEKLY_HTML
snapshot_path = SNAPSHOT_CSV

try:
    monitoring_df = load_monitoring_frame(
        source="mssql",
        table=VITRINA_TABLE_DEFAULT,
    )
    if not RETRO_PARQUET.exists():
        raise FileNotFoundError(
            f"Не найден финальный ретро-датасет: {RETRO_PARQUET}"
        )
    retro_df = pd.read_parquet(RETRO_PARQUET)
    result = estimate_monitoring_effect(
        monitoring_df,
        retro_df,
        t_calc=T_CALC,
        residual_share=RESIDUAL_SHARE,
        discount_rate=DISCOUNT_RATE,
        lookback_years=LOOKBACK_YEARS,
        retro_as_of=RETRO_AS_OF,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        bootstrap_compliance_iterations=BOOTSTRAP_COMPLIANCE_ITERATIONS,
        bootstrap_n_jobs=BOOTSTRAP_N_JOBS,
    )
    development_lags = None
    pret = (
        pd.read_parquet(PRETENSIONS_PARQUET)
        if PRETENSIONS_PARQUET.exists()
        else None
    )
    claims = pd.read_parquet(CLAIMS_PARQUET) if CLAIMS_PARQUET.exists() else None
    if pret is not None or claims is not None:
        development_lags = compute_psr_development_lags(
            pretensions=pret,
            claims=claims,
            incidents=retro_df,
        )
    plan_path, report_path, conclusion_path, artifact_names = write_all_monitoring_htmls(
        result,
        DATA_DIR,
        source_label=VITRINA_TABLE_DEFAULT,
        development_lags=development_lags,
        stamp=export_stamp,
    )
    if WRITE_AUDIT_XLSX:
        audit_path = export_monitoring_audit_xlsx(
            result, run_dir / artifact_names["audit"]
        )
    snapshot_path = append_snapshot_log(
        result,
        SNAPSHOT_CSV,
        source_label=VITRINA_TABLE_DEFAULT,
        note=f"full run stamp={export_stamp}",
    )
    if RUN_WEEKLY_SERIES:
        weekly_series = run_weekly_monitoring_series(
            monitoring_df,
            retro_df,
            pilot_start=PILOT_START,
            as_of=result.t_calc,
            residual_share=RESIDUAL_SHARE,
            discount_rate=DISCOUNT_RATE,
            lookback_years=LOOKBACK_YEARS,
            retro_as_of=RETRO_AS_OF,
            bootstrap_iterations=WEEKLY_BOOTSTRAP_ITERATIONS,
            bootstrap_compliance_iterations=0,
            bootstrap_n_jobs=1,
        )
        weekly_html = save_weekly_outputs(weekly_series, DATA_DIR)
except Exception as exc:
    report_path = write_error_html(
        exc,
        run_dir / artifact_names["error"],
        source_label=VITRINA_TABLE_DEFAULT,
        plan_name=artifact_names["plan"],
        report_name=artifact_names["report"],
        conclusion_name=artifact_names["conclusion"],
    )
    raise
finally:
    print(f"stamp / run_dir → {export_stamp} / {run_dir}")
    print(f"План → {plan_path}")
    print(f"Расчёт → {report_path}")
    print(f"Заключение → {conclusion_path}")
    print(f"Excel-аудит → {audit_path}")
    print(f"Снимки → {snapshot_path}")
    print(f"Понедельно → {weekly_html}")
